# Claude Code Token Reduction Test — Runbook

**Goal:** test whether five context/token-reduction tools — **Graphify** (graph indexing),
**Repomix** (repo packing), **Caveman** (input/output compression), **Ponytail**
(minimal-code-generation), and **Context Mode** (context sandboxing) — reduce Claude Code
token usage on the `poc-early-warning` repo, and by how much, each independently against baseline.

Run the cells **in order, top to bottom**. Cells you'll re-run many times (Section 5's run loop)
are marked. Where a judgment call was already made for you it's stated as fact; where you still
need to decide something it's marked **DECIDE:**.

**Design note:** this is a one-arm-per-tool test (baseline + 5 tool arms), not a combinatorial
one. The five tools work by different mechanisms and stacking them would confound which tool
caused which change. Run a combined-tool test as a follow-up once you have clean single-tool
numbers, using this same notebook as a template.


## 0. One-time setup

### 0.0 Freeze the Claude Code version and model — do this FIRST

This is the answer to "is there a way to freeze which version and model we're using": yes, and
it needs three separate things, not just picking a model in `/model` once (that setting doesn't
survive well across fresh headless sessions, and Claude Code auto-updates the CLI itself in the
background by default, so "the same tool" can silently mean a different build on day 3 than day 1).

1. **Pin the CLI binary version** — install one exact version instead of "latest".
2. **Disable background auto-update** — so it can't drift mid-test.
3. **Pass `--model <exact-string>` on every single invocation** — don't rely on an interactive
   `/model` pick persisting into headless (`-p`) runs, which is how every run in this notebook
   invokes Claude Code.

Fill in the two values below, then run this cell. If you don't know the current version yet,
run `!claude --version` in a scratch cell first.


In [1]:
import os, subprocess

# --- DECIDE: fill these in and don't change them until the whole test is done ---
CLAUDE_PINNED_VERSION = "2.1.211"          # exact version string from `claude --version`
CLAUDE_PINNED_MODEL   = "claude-sonnet-5"  # exact model string, never an alias like "sonnet"
REPO_DIR              = os.path.expanduser("~/poc-early-warning")
# ----------------------------------------------------------------------------------

os.environ["CLAUDE_PINNED_VERSION"] = CLAUDE_PINNED_VERSION
os.environ["CLAUDE_PINNED_MODEL"] = CLAUDE_PINNED_MODEL
os.environ["REPO_DIR"] = REPO_DIR

print(f"Pinned version: {CLAUDE_PINNED_VERSION}")
print(f"Pinned model:   {CLAUDE_PINNED_MODEL}")
print(f"Repo dir:       {REPO_DIR}")


Pinned version: 2.1.211
Pinned model:   claude-sonnet-5
Repo dir:       /Users/rjw/poc-early-warning


In [2]:
%%bash
# Install the exact pinned CLI version (not "latest") and disable background auto-update.
curl -fsSL https://claude.ai/install.sh | bash -s "$CLAUDE_PINNED_VERSION"

mkdir -p ~/.claude
python3 - <<'PY'
import json, os
path = os.path.expanduser("~/.claude/settings.json")
data = {}
if os.path.exists(path):
    with open(path) as f:
        data = json.load(f)
data.setdefault("env", {})["DISABLE_AUTOUPDATER"] = "1"
with open(path, "w") as f:
    json.dump(data, f, indent=2)
print("DISABLE_AUTOUPDATER=1 set in", path)
PY

claude --version
claude doctor | grep -i "auto-update" || true


Setting up Claude Code...

Checking installation status...
Installing Claude Code native build 2.1.211...
Setting up launcher and shell integration...
✔ Claude Code successfully installed!

  Version: 2.1.211

  Location: ~/.local/bin/claude


  Next: Run claude --help to get started

✅ Installation complete!

DISABLE_AUTOUPDATER=1 set in /Users/rjw/.claude/settings.json
2.1.211 (Claude Code)
Auto-updates: disabled (set by env: DISABLE_AUTOUPDATER)
Auto-update channel: latest


In [3]:
%%bash
# Guard: fail loudly if the installed version doesn't match the pin. Run this at the start
# of every work session, not just once — someone could still run a bare `claude update`.
ACTUAL=$(claude --version | awk '{print $1}')
if [ "$ACTUAL" != "$CLAUDE_PINNED_VERSION" ]; then
  echo "MISMATCH: installed claude is $ACTUAL, pinned version is $CLAUDE_PINNED_VERSION" >&2
  exit 1
fi
echo "OK: claude $ACTUAL matches pin"


OK: claude 2.1.211 matches pin


From here on, every `claude` invocation in this notebook goes through the
`run_claude_headless()` helper defined in 0.1 below, which always passes
`--model "$CLAUDE_PINNED_MODEL"` explicitly — so the model is frozen per-call, not per-session,
and can't drift even if a session gets resumed or a default changes.


### 0.1 Headless-invocation helper, then clone the repo and confirm it runs

`claude -p "<prompt>"` runs one turn to completion non-interactively and exits — that's what
makes this test scriptable at all. Every call in this notebook (installing plugins, toggling
tools, running the actual test tasks) goes through this one function so the pinned model and
system prompt are never accidentally left off a call.


In [4]:
import subprocess, os

def run_claude_headless(prompt: str, cwd: str = None, timeout: int = 1800) -> subprocess.CompletedProcess:
    """Run one headless Claude Code turn with the pinned model and the fixed
    noise-reduction system prompt (set in 0.4) always attached."""
    cmd = [
        "claude",
        "--model", os.environ["CLAUDE_PINNED_MODEL"],
        "--append-system-prompt", os.environ.get("SYSTEM_PROMPT", ""),
        "-p", prompt,
    ]
    return subprocess.run(cmd, cwd=cwd or os.environ["REPO_DIR"],
                           capture_output=True, text=True, timeout=timeout)


In [7]:
%%bash
git clone https://github.com/rjwdata/poc-early-warning.git "$REPO_DIR" 2>/dev/null || echo "already cloned"
cd "$REPO_DIR" && uv venv && source .venv/bin/activate && make install-dev && make train


already cloned


Using CPython 3.10.18
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


Installing development dependencies with uv...
uv pip install -e ".[dev,test]"


Resolved 185 packages in 144ms
   Building poc-early-warning @ file:///Users/rjw/poc-early-warning
      Built poc-early-warning @ file:///Users/rjw/poc-early-warning
Prepared 1 package in 799ms
Uninstalled 1 package in 1ms
Installed 1 package in 2ms
 ~ poc-early-warning==0.1.0 (from file:///Users/rjw/poc-early-warning)


Development installation complete!
Training model...
uv run python src/components/data_ingestion_eda.py


Traceback (most recent call last):
  File "/Users/rjw/poc-early-warning/src/components/data_ingestion_eda.py", line 65, in initiate_data_ingestion
    df = pd.read_csv(self.ingestion_config.raw_data_path)
  File "/Users/rjw/poc-early-warning/.venv/lib/python3.10/site-packages/pandas/io/parsers/readers.py", line 1024, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/Users/rjw/poc-early-warning/.venv/lib/python3.10/site-packages/pandas/io/parsers/readers.py", line 618, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/Users/rjw/poc-early-warning/.venv/lib/python3.10/site-packages/pandas/io/parsers/readers.py", line 1618, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "/Users/rjw/poc-early-warning/.venv/lib/python3.10/site-packages/pandas/io/parsers/readers.py", line 1878, in _make_engine
    self.handles = get_handle(
  File "/Users/rjw/poc-early-warning/.venv/lib/python3.10/site-packages/pandas/io/common.py", line 873, in g

CalledProcessError: Command 'b'git clone https://github.com/rjwdata/poc-early-warning.git "$REPO_DIR" 2>/dev/null || echo "already cloned"\ncd "$REPO_DIR" && uv venv && source .venv/bin/activate && make install-dev && make train\n'' returned non-zero exit status 2.

In [ ]:
%%bash
cd "$REPO_DIR" && git checkout . && git clean -fd


### 0.2 CLAUDE.md handling — DECIDED: remove it for the test

This repo ships a `CLAUDE.md` with the architecture already written out. Leaving it in place
means Claude Code auto-loads it every session in every arm, so the baseline arm already knows
the answer to most read-heavy tasks and none of the tools' real effect will show. This also
matters specifically for Graphify, which can write its own `CLAUDE.md` back in "always-on" mode
— see Arm B below.

Keep `CLAUDE.md.hold` — don't delete it, you'll move it back at the very end (Section 8).


In [ ]:
%%bash
cd "$REPO_DIR"
[ -f CLAUDE.md ] && mv CLAUDE.md CLAUDE.md.hold || echo "CLAUDE.md already moved or absent"


### 0.3 Fixed system-prompt flag — set once, use for every single run

Not part of the per-arm toggle logic — a constant that applies identically to every arm so it
can't bias the comparison. Without it, the agent can occasionally run an unscoped repo-wide
`find`/`grep` that sweeps `.venv` (thousands of site-packages files), producing multi-megabyte
tool output for reasons unrelated to any of the five tools.


In [ ]:
import os
os.environ["SYSTEM_PROMPT"] = (
    "When listing or searching the repository, always exclude .venv, .git, node_modules, "
    ".graphify-out, .repomix, and any tool-specific cache or index directory."
)
print(os.environ["SYSTEM_PROMPT"])


### 0.4 Install the five tools' binaries/plugins (do NOT activate any of them yet)

"Install" here means "get the binary/plugin on disk and confirm its version" — none of these
commands should make the tool participate in a Claude Code session yet. Activation is per-arm
(Section 4).


In [ ]:
%%bash
# Graphify — index/graph tool. Binary only; do NOT run `graphify install` or `/graphify .` yet.
uv tool install graphifyy && graphify --version


In [ ]:
%%bash
# Repomix — repo-packing tool. CLI only; do NOT register the MCP/plugin yet.
npm install -g repomix && repomix --version


In [ ]:
%%bash
# Caveman — compression proxy + skill. The installer wires itself into Claude Code
# automatically and may leave itself live, so turn it off immediately after installing.
curl -fsSL https://raw.githubusercontent.com/JuliusBrussee/caveman/v2.2.0/install.sh | bash
caveman --version
caveman off 2>/dev/null; true


In [ ]:
# Ponytail and Context Mode are Claude Code plugins. `marketplace add` only registers the
# marketplace, not the plugin itself -- installing the plugin is deferred to each one's arm
# (Section 4). These are slash commands run *inside* a session, so they go through the
# headless helper rather than a shell pipeline.
r1 = run_claude_headless("/plugin marketplace add DietrichGebert/ponytail")
print(r1.stdout, r1.stderr)
r2 = run_claude_headless("/plugin marketplace add mksglu/context-mode")
print(r2.stdout, r2.stderr)


### 0.5 Verify nothing is active yet

`trace_settings.py` (shipped alongside this notebook) automates the "check every surface, not
just the obvious one" step: it loads every `settings.json` it can find, validates the parts it
understands (hooks, permissions) with pydantic, AND does a generic recursive walk of the raw
JSON so nothing in an unmodeled field — a plugin-registration key, an env var, anything — gets
missed. It exits 1 if anything mentioning one of the five tools turns up.


In [ ]:
%%bash
cd "$REPO_DIR"
python3 /mnt/user-data/outputs/trace_settings.py --project-dir "$REPO_DIR"
claude mcp list
claude plugin list 2>/dev/null || true
graphify hook status 2>/dev/null || true
caveman stats 2>/dev/null || true
ls .graphify-out 2>/dev/null || true
ls CLAUDE.md 2>/dev/null && echo "WARNING: CLAUDE.md exists -- did something write it back?"
ps aux | grep -i caveman | grep -v grep || true


Anything flagged is a leftover — resolve it by hand (uninstall commands are in each
tool's arm section, Section 4) before running Arm A.


### 0.6 Run-logging script

`track_run.py` (also shipped alongside this notebook) removes manual token/duration
bookkeeping — pasting numbers off `/cost` by hand is exactly the kind of step that introduces
run-to-run inconsistency. It has `start`/`finish` subcommands and now takes `--tool` so it knows
which of the five tools' "did it actually fire" signals to check (see the script's own
docstring for the signal each tool maps to). It also reads the CLI version + model straight out
of each transcript and flags a mismatch against `CLAUDE_PINNED_VERSION`/`CLAUDE_PINNED_MODEL`.


In [ ]:
%%bash
chmod +x /mnt/user-data/outputs/track_run.py
python3 /mnt/user-data/outputs/track_run.py --help


## 1. The task set (already built — just use it)

Copy-paste each task prompt **exactly as written** every time. Don't paraphrase, don't add
detail, don't fix typos — consistency matters more than polish. Stored below as a Python list
so later cells can run them programmatically.

This is a real, verified bug we found (not hypothetical, relevant to task 8): `config/params.yaml`
has a whole `KNNImputer` block sitting unused, and the original `CLAUDE.md` (now moved aside)
documented `KNNImputer` as if it were live — but the actual code silently uses
`SimpleImputer(median)` instead.


In [ ]:
TASKS = [
    {
        "num": 1, "category": "Read-heavy",
        "prompt": "Map the repo structure: identify the Streamlit app entry point, the multi-page UI, the ML pipeline stages, and where trained artifacts live.",
        "criteria": "Names app_main.py as entry point, pages/01-05_*.py as the UI, the three pipeline stages in src/components/ (data_ingestion_eda.py -> data_transformation.py -> model_trainer.py), and artifacts/ for outputs.",
    },
    {
        "num": 2, "category": "Read-heavy",
        "prompt": "Trace the ML pipeline end-to-end: from raw data through preprocessing, training, to how a prediction actually gets served in the Streamlit app.",
        "criteria": "Traces raw CSV -> ingestion/split -> preprocessing (numeric: SimpleImputer(strategy='median') -> StandardScaler; categorical: OneHotEncoder) -> training -> artifacts/preprocessor.pkl + artifacts/model.pkl -> src/pipeline/predict_pipeline.py -> pages/03_predictions.py. Note: the real code uses SimpleImputer, not KNNImputer -- that's correct, not wrong.",
    },
    {
        "num": 3, "category": "Read-heavy",
        "prompt": "Inventory the data quality / model evaluation surfaces in this repo -- what gets checked, and where results are surfaced.",
        "criteria": "Identifies Evidently AI quality reports (artifacts/*.html), the pages/05_data_quality.py viewer, and the model comparison/fairness content in pages/02_technical_details.py and pages/04_model_cards.py.",
    },
    {
        "num": 4, "category": "Shell-heavy",
        "prompt": "Stand up the local dev environment from a clean checkout and attempt to run the test suite.",
        "criteria": "Runs uv venv && make install-dev successfully; accurately reports that make test fails because there is no tests/ directory. Reporting this accurately = pass.",
    },
    {
        "num": 5, "category": "Shell-heavy",
        "prompt": "Run the full training pipeline locally (make train) and confirm the expected artifacts are produced.",
        "criteria": "artifacts/model.pkl and artifacts/preprocessor.pkl (re)generated without error.",
    },
    {
        "num": 6, "category": "Shell-heavy",
        "prompt": "Run the linters (make lint) and fix any warnings they raise.",
        "criteria": "ruff check src/, black --check src/, mypy src/ all pass, or agent accurately reports what it couldn't safely auto-fix.",
    },
    {
        "num": 7, "category": "Mixed",
        "prompt": "Add a first test case (create a tests/ directory if it doesn't exist) covering data_transformation.py's preprocessing pipeline.",
        "criteria": "New test file runs under pytest, meaningfully exercises the ColumnTransformer logic (not a placeholder assert True).",
    },
    {
        "num": 8, "category": "Mixed",
        "prompt": "The config/params.yaml file defines a KNNImputer configuration block (n_neighbors: 3, weights: uniform), but src/components/data_transformation.py never reads config/params.yaml at all -- it hardcodes SimpleImputer(strategy='median') instead. Fix this: wire data_transformation.py to actually read the imputer settings from config/params.yaml and use KNNImputer as configured.",
        "criteria": "data_transformation.py now reads config/params.yaml, uses KNNImputer with the configured n_neighbors/weights, make train still runs successfully end-to-end.",
    },
    {
        "num": 9, "category": "Mixed",
        "prompt": "Add one new predictive feature to the training pipeline (a derived column of your choice) and verify it flows through to the pages/03_predictions.py input form and prediction output.",
        "criteria": "New feature is added to the feature set, model_trainer.py trains with it included, prediction page accepts/uses it without breaking existing fields.",
    },
]
for t in TASKS:
    print(t["num"], t["category"], "-", t["prompt"][:70], "...")


## 2. The six test arms

| Arm | Tool | Mechanism |
|---|---|---|
| A — Baseline | none | -- |
| B — Graphify | graphify | tree-sitter AST index + graph, queried instead of re-reading files |
| C — Repomix | repomix | packs the repo into one AI-optimized file/MCP tool |
| D — Caveman | caveman | compresses agent input/output via local proxy + terse-response mode |
| E — Ponytail | ponytail | enforces minimal code generation via a prioritization ladder |
| F — Context Mode | context-mode | sandboxes tool output, persists session memory, routes via MCP + hooks |

**Run the arms in this order: A, B, C, D, E, F.** Do all 3 repeats of all 9 tasks for one arm
before moving to the next — don't interleave. Only one arm's tool should ever be active at a
time.


In [ ]:
ARMS = {
    "A": "none",
    "B": "graphify",
    "C": "repomix",
    "D": "caveman",
    "E": "ponytail",
    "F": "context-mode",
}


## 3. Repo reset procedure — runs automatically before every task run in Section 5

Does **not** touch a tool's index/cache directory or a global hook/proxy config -- those are
toggled once per arm (Section 4), not per run.


In [ ]:
def reset_repo():
    subprocess.run(["git", "checkout", "."], cwd=os.environ["REPO_DIR"], check=True)
    subprocess.run(["git", "clean", "-fd"], cwd=os.environ["REPO_DIR"], check=True)


## 4. Arm-by-arm toggle functions

Each tool has a different mechanism, so each gets its own on/off pair rather than one generic
function. `arm_on("B")` and `arm_off("B")` (etc.) are what you call between batches of runs in
Section 5.


In [ ]:
def arm_on(arm: str):
    tool = ARMS[arm]
    repo = os.environ["REPO_DIR"]
    if tool == "none":
        subprocess.run(["claude", "mcp", "list"])
    elif tool == "graphify":
        run_claude_headless("/graphify .", cwd=repo)
        subprocess.run(["graphify", "claude", "install"], cwd=repo, check=True)
        subprocess.run(["graphify", "hook", "status"], cwd=repo)
    elif tool == "repomix":
        subprocess.run(["claude", "mcp", "add", "repomix", "--", "npx", "-y", "repomix", "--mcp"], check=True)
        subprocess.run(["claude", "mcp", "list"])
    elif tool == "caveman":
        subprocess.run(["caveman", "full"], check=True)
        subprocess.run(["caveman", "stats"])
    elif tool == "ponytail":
        r = run_claude_headless("/plugin install ponytail@ponytail")
        print(r.stdout, r.stderr)
        r = run_claude_headless("/ponytail full")
        print(r.stdout, r.stderr)
    elif tool == "context-mode":
        r = run_claude_headless("/plugin install context-mode@context-mode")
        print(r.stdout, r.stderr)
        r = run_claude_headless("/context-mode:ctx-doctor")
        print(r.stdout, r.stderr)
    else:
        raise ValueError(tool)
    print(f"Arm {arm} ({tool}) is ON. Run the Step 0.5-style surface check before trusting this.")


def arm_off(arm: str):
    tool = ARMS[arm]
    repo = os.environ["REPO_DIR"]
    if tool == "none":
        pass
    elif tool == "graphify":
        subprocess.run(["graphify", "claude", "uninstall"], cwd=repo)
        subprocess.run(["graphify", "uninstall", "--purge"], cwd=repo)
        claude_md = os.path.join(repo, "CLAUDE.md")
        if os.path.exists(claude_md):
            print("WARNING: Graphify wrote CLAUDE.md back -- remove it before the next arm.")
    elif tool == "repomix":
        subprocess.run(["claude", "mcp", "remove", "repomix"])
    elif tool == "caveman":
        subprocess.run(["caveman", "off"])
        print("Confirm the background proxy actually stopped: `ps aux | grep -i caveman`")
    elif tool == "ponytail":
        r = run_claude_headless("/plugin remove ponytail")
        print(r.stdout, r.stderr)
    elif tool == "context-mode":
        r = run_claude_headless("/plugin uninstall context-mode@context-mode")
        print(r.stdout, r.stderr)
    else:
        raise ValueError(tool)
    print(f"Arm {arm} ({tool}) is OFF.")


⚠️ **Caveman's proxy is global to your machine**, not scoped to this repo -- like rtk's
hook was in the earlier version of this test. Don't do unrelated Claude Code work during Arm D,
and confirm with `ps aux | grep -i caveman` that the process actually stopped when you call
`arm_off("D")`, not just that the skill mode reports off.

**Context Mode activates the moment it's installed** (via a `SessionStart` hook) -- there's no
separate index-build step the way Graphify has, so `arm_on("F")` *is* the activation.


## 5. Executing runs (9 tasks × 6 arms × 3 repeats = 162 total)

`run_one()` does one full run: reset repo, snapshot the tool signal, invoke Claude Code
headlessly with the task prompt, print the transcript's final response so you can score it, ask
you for pass/partial/fail, then log the row via `track_run.py finish`.

Run `arm_on(arm)` once per arm, then call `run_one()` for each task × repeat in that arm, then
`arm_off(arm)` before moving to the next arm's `arm_on()`.


In [ ]:
import re

def run_one(task: dict, arm: str, repeat: int, csv_path: str = "results.csv"):
    tool = ARMS[arm]
    reset_repo()
    subprocess.run(["python3", "/mnt/user-data/outputs/track_run.py", "start", "--tool", tool], check=True)

    result = run_claude_headless(task["prompt"])
    print("----- Claude's final response -----")
    print(result.stdout[-4000:])  # tail, in case it's long
    if result.returncode != 0:
        print("----- stderr -----")
        print(result.stderr[-2000:])

    print(f"\nScoring criteria for task {task['num']}: {task['criteria']}")
    score = input("Result (pass/partial/fail): ").strip().lower()
    while score not in ("pass", "partial", "fail"):
        score = input("Please type pass, partial, or fail: ").strip().lower()

    subprocess.run([
        "python3", "/mnt/user-data/outputs/track_run.py", "finish",
        "--task", str(task["num"]), "--arm", arm, "--tool", tool,
        "--repeat", str(repeat), "--result", score,
        "--csv", csv_path, "--project-dir", os.environ["REPO_DIR"],
    ], check=True)


In [ ]:
def run_arm(arm: str, repeats: int = 3, tasks=None, csv_path: str = "results.csv"):
    """Runs every task the given number of times for one arm. Call arm_on(arm)
    yourself first and arm_off(arm) yourself after -- kept separate so you can
    inspect the arm's activation before committing 27 runs to it."""
    for task in (tasks or TASKS):
        for repeat in range(1, repeats + 1):
            print(f"\n=== Arm {arm} ({ARMS[arm]}) / Task {task['num']} / Repeat {repeat} ===")
            run_one(task, arm, repeat, csv_path=csv_path)


### Arm A — Baseline

Run these three cells in order (verify -> run -> off), then repeat the same on/run/off pattern
for arms B–F using their own toggles from Section 4.


In [ ]:
arm_on("A")
python_check = subprocess.run(["python3", "/mnt/user-data/outputs/trace_settings.py", "--project-dir", os.environ["REPO_DIR"]])
assert python_check.returncode == 0, "Resolve findings before running Arm A"


In [ ]:
run_arm("A", repeats=3)

In [ ]:
arm_off("A")

### Arm B — Graphify

In [ ]:
arm_on("B")

In [ ]:
run_arm("B", repeats=3)

In [ ]:
arm_off("B")

### Arm C — Repomix

In [ ]:
arm_on("C")

In [ ]:
run_arm("C", repeats=3)

In [ ]:
arm_off("C")

### Arm D — Caveman

In [ ]:
arm_on("D")

In [ ]:
run_arm("D", repeats=3)

In [ ]:
arm_off("D")

### Arm E — Ponytail

In [ ]:
arm_on("E")

In [ ]:
run_arm("E", repeats=3)

In [ ]:
arm_off("E")

### Arm F — Context Mode

In [ ]:
arm_on("F")

In [ ]:
run_arm("F", repeats=3)

In [ ]:
arm_off("F")

## 6. Tracking

`results.csv` now has one row per run (162 total if every arm ran clean). Load it for a quick
sanity check before handing off for the real analysis.


In [ ]:
import pandas as pd
df = pd.read_csv("results.csv")
print(len(df), "rows")
df.groupby("arm")[["total_tokens", "duration_seconds"]].median()


## 7. After all 162 runs: hand back to Claude

Share `results.csv` plus 2-3 sample transcript JSONL files (paths are in the
`transcript_path` column), one from each arm, in a new conversation. From there: median %
token reduction per arm vs. baseline, broken out by task category, plus the tool-call-type
breakdown that checks whether the savings are plausible given what each tool can actually
touch. That analysis needs the real data — don't try to eyeball it from the CSV alone.


## 8. Final cleanup (after the whole test is done, all arms)

In [ ]:
%%bash
cd "$REPO_DIR"
claude mcp remove repomix 2>/dev/null || true
graphify uninstall --purge 2>/dev/null || true
caveman off 2>/dev/null || true


In [ ]:
r1 = run_claude_headless("/plugin uninstall ponytail@ponytail")
print(r1.stdout, r1.stderr)
r2 = run_claude_headless("/plugin uninstall context-mode@context-mode")
print(r2.stdout, r2.stderr)


In [ ]:
%%bash
cd "$REPO_DIR"
[ -f CLAUDE.md.hold ] && mv CLAUDE.md.hold CLAUDE.md
git checkout . && git clean -fd
python3 /mnt/user-data/outputs/trace_settings.py --project-dir "$REPO_DIR"


## Gotchas — read this before you start, not after something goes wrong

- **These five tools don't share one install pattern.** Graphify is a CLI you build an index
  with (`/graphify .`) and optionally make always-on. Repomix is a CLI/MCP hybrid. Caveman is a
  global proxy plus an in-session skill. Ponytail and Context Mode are Claude Code plugins
  installed via `/plugin marketplace add` + `/plugin install`. Always use that tool's own
  `arm_on`/`arm_off` branch in Section 4, never assume one tool's toggle works for another.
- **Plugin-install and skill-toggle commands are slash commands run inside a Claude Code
  session**, which is why `arm_on`/`arm_off` route them through `run_claude_headless()` rather
  than a plain shell call. This wasn't independently verified for every plugin's exact headless
  behavior — check the printed stdout/stderr from those cells before trusting an arm is on.
- **Graphify's "always-on" mode can write its own `CLAUDE.md`.** Since Section 0.2 removed the
  repo's original `CLAUDE.md` specifically so no arm gets extra context the others don't, check
  for a regenerated one after every Graphify run.
- **Caveman's proxy is global to your machine.** Run Arm D in a dedicated block of time and
  confirm with `ps aux | grep -i caveman` that it actually stopped when you call `arm_off("D")`.
- **"Config toggled on" is not proof a tool fired during a given run.** `track_run.py`'s
  `tool_signal_changed` column is exactly this check — an empty diff on an arm where the tool
  should be active means the toggle didn't reflect reality for that run. Flag it, don't trust it
  as a clean data point.
- **The version/model pin (Section 0.0) is fixed across all six arms.** `track_run.py` reads the
  actual version/model out of each transcript and will print a `version_model_mismatch` warning
  if a run drifted from the pin — check `results.csv` for a non-empty value in that column
  before trusting the run.
- **If a run fails the task, log it anyway.** A fast failure with low token count is not a win.
- **`track_run.py` auto-detects the transcript by most-recently-modified file that isn't
  already logged.** If you run another Claude Code session for something unrelated mid-test
  before calling `finish`, pass `--transcript PATH` explicitly (or, in this notebook, avoid
  running unrelated sessions during the test window at all).
